# Haicai-v1 RL on Colab T4 — Verifiers v1 + Unsloth + TRL GRPO

**One-click GRPO for Brazilian-portuguese haiku (5-7-5).** Open-source, deterministic, free-T4.

This is the clean rebuild: the haiku engine is **not vendored** inside the notebook — it's the `haicai` pip package (escansão interval + AO90 vocab) and the `haicai-v1` verifiers taskset, both installed directly from GitHub. The notebook only glues model + dataset + reward, so the reward *is* the env.

| layer | what | VRAM |
|---|---|---|
| `haicai` | `escandir` interval + `orto.coverage` over 40k AO90 vocab, 2µs/verso | 0 |
| `haicai-v1` | `verifiers.v1` taskset (`HaicaiTask` `@reward`/`@metric`, `HaicaiTaskset.load`) | 0 |
| `verifiers` | `Trace` + `Task.score()` (no `prime` runtime, no sandbox) | 0 |
| `TRL GRPO` | group baseline, `beta=0.0`, `bnpo` loss | — |
| `Unsloth` | 2× LoRA, T4-friendly | 5–10GB |

> Validated: 11/11 Guilherme de Almeida, 16/16 external gold, 115k-verse smoke (0 crashes, `0.3ms/verso`), 48/48 ear check (`PROCESS.md`). Qwen3.5 GRPO path verified against Unsloth `2026.8.23` / `fix#5880` (T4 bf16→fp16) + `issues/4970` (QLoRA for T4) — Qwen3.5-0.8B needs `load_in_4bit=True` on T4.

> **Models:** `Qwen3.5-0.8B` (post-trained; saucy, ~3GB) is the default bootstrap. `Qwen3.5-0.8B-Base` is the separate pre-trained checkpoint. Qwen3.5 keeps its native tokenizer/chat template; this notebook does not replace it with the older `qwen3` template. `Qwen3-4B` also works with `transformers 4.56.2`.


In [ ]:
%%capture
#@title 1 — Install (run once, then Runtime → Restart if prompted)
import os, importlib.util, sys, pathlib
REPO_URL = "https://github.com/ob1-s/haicai.git"
ENGINE_URL = "https://github.com/ob1-s/escansao.git"
REPO_DIR = "/content/haicai"
!pip install -q uv
# Clear stale Unsloth compile cache (fixes dtype mismatch after version bumps)
!rm -rf unsloth_compiled_cache
# Torch/Triton only on fresh colab; skip on re-runs to save time
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try:
        import numpy, PIL
        _numpy, _pil = f"numpy=={numpy.__version__}", f"pillow=={PIL.__version__}"
    except Exception:
        _numpy, _pil = "numpy", "pillow"
    !uv pip install -q "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -q unsloth
# Qwen3.5 needs transformers>=5.5 on T4; Qwen3 works with 4.56.2 — we pin 5.5.0 for 3.5 (saucy)
!uv pip install -q --no-deps "transformers>=5.5.0" "tokenizers>=0.22.0,<=0.23.0" "trl>=0.28.0" unsloth unsloth_zoo
!uv pip install -q verifiers "datasets>=5.0.0" "huggingface_hub>=0.34.0" hf_transfer wandb protobuf accelerate peft nest_asyncio
# — haicai open-source (single source of truth) — clone once, install —
if not pathlib.Path(REPO_DIR).exists():
    !git clone {REPO_URL} {REPO_DIR} 2>&1 | tail -3
# Engine via git until `escansao` lands on PyPI (then: `!uv pip install -q escansao`)
!uv pip install -q "escansao @ git+{ENGINE_URL}" 2>&1 | tail -2
!uv pip install -q --no-deps -e {REPO_DIR}/environments/haicai_v1 2>&1 | tail -2
# Fallback: if still not importable (e.g. need kernel restart), expose via sys.path without restart
import sys
for _p in [f"{REPO_DIR}/environments/haicai_v1"]:
    if _p not in sys.path:
        sys.path.insert(0, _p)
import importlib
for _m in ["haicai", "haicai_v1"]:
    print(_m, importlib.util.find_spec(_m))


## 2 — Config
Smoke = 64 tasks × G=4 × 10 steps (~10 min on T4). For a real run set `num_tasks=400`, `max_steps=80`, `rollouts_per_sample=8`.


In [ ]:
#@title 2 — Config
# Qwen3.5-0.8B (no suffix) is post-trained and is the default bootstrap for PT haiku. Keep its native Qwen3.5 chat template.
model_id = "unsloth/Qwen3.5-2B" #@param ["unsloth/Qwen3.5-0.8B", "unsloth/Qwen3.5-0.8B-Base", "unsloth/Qwen3.5-1.5B", "unsloth/Qwen3-4B", "unsloth/Qwen2.5-1.5B-Instruct", "unsloth/gemma-3-1b-it"] {allow-input:true}
max_seq_length = 384 #@param {type:"integer"}
lora_rank = 32 #@param {type:"integer"}
# Qwen3.5 on T4 MUST use 4-bit (bf16→Half hell #4970, fix #5880) — we auto-force it below
load_in_4bit = True #@param {type:"boolean"}
use_wandb = False #@param {type:"boolean"}
wandb_project = "haicai-v1-colab" #@param {type:"string"}
seed = 3301 #@param {type:"integer"}

# Taskset slice (train/test split by offset)
num_tasks = 64 #@param {type:"integer"}
task_offset = 0 #@param {type:"integer"}
eval_tasks = 16 #@param {type:"integer"}
eval_offset = 1000 #@param {type:"integer"}

# GRPO
max_completion_length = 64 #@param {type:"integer"}
temperature = 1.0 #@param {type:"number"}
learning_rate = 5e-6 #@param {type:"number"}
rollouts_per_sample = 16 #@param {type:"integer"}
max_steps = 80 #@param {type:"integer"}
save_steps = 20 #@param {type:"integer"}
enable_thinking = False #@param {type:"boolean"}


In [ ]:
#@title 3 — Load model (Unsloth) — Qwen3.5 T4 fix
import torch
# Fix Unsloth Zoo Gemma-4 proxy (#6089) — no-op for Qwen
try:
    import unsloth_zoo.temporary_patches.gemma4 as _g4
    _proxy = getattr(_g4, "_Gemma4KVSharedSafeProxy", None)
    if _proxy is not None and not getattr(_proxy, "_haicai_patched", False):
        _orig = _proxy.__getattr__
        def _patched(self, name):
            return 0 if name == "num_kv_shared_layers" else _orig(self, name)
        _proxy.__getattr__ = _patched
        _proxy._haicai_patched = True
except Exception as e:
    print(f"proxy patch skip: {e}")

# Qwen3.5 on T4: must use 4-bit QLoRA (issues/4970) — auto-force for 3.5, respect toggle for others
is_qwen35 = "qwen3" in model_id.lower() and "3.5" in model_id
is_gemma4 = "gemma-4" in model_id.lower()
effective_4bit = True if is_qwen35 else (load_in_4bit if not is_gemma4 else True)
if is_qwen35 and not effective_4bit:
    print("Qwen3.5 on T4: forcing load_in_4bit=True (bf16→Half fix #4970)")
    effective_4bit = True

if is_gemma4:
    from unsloth import FastVisionModel as Loader
else:
    from unsloth import FastLanguageModel as Loader

model, tokenizer = Loader.from_pretrained(
    model_name=model_id,
    max_seq_length=max_seq_length,
    load_in_4bit=effective_4bit,
    fast_inference=False,  # required for GRPO (Unsloth)
)
# Keep each checkpoint's native tokenizer/chat template. In particular, do not overwrite Qwen3.5 with the older `qwen3` template.

model = Loader.get_peft_model(
    model, r=lora_rank, lora_alpha=lora_rank*2,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=seed,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Loaded {model_id}  4bit={effective_4bit}  lora={lora_rank}  seq={max_seq_length}")
if torch.cuda.is_available():
    print(f"GPU {torch.cuda.get_device_name(0)}  VRAM={torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")
if use_wandb:
    import wandb, os; os.environ["WANDB_PROJECT"] = wandb_project


In [ ]:
#@title 4 — Load haicai-v1 (hub-style, single source) → HF Dataset
import sys, pathlib
# Fallback if Install cell didn't trigger import (no restart): ensure haicai is on path
for _p in ["/content/haicai/environments/haicai_v1"]:
    if pathlib.Path(_p).exists() and _p not in sys.path:
        sys.path.insert(0, _p)

from datasets import Dataset
try:
    from haicai_v1.taskset import HaicaiTaskset, HaicaiConfig
    from escansao import escandir
    from haicai_v1.orto import coverage
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        f"{e}. Tried sys.path={sys.path[:3]}. "
        "Fix: re-run Install cell, then Runtime → Restart session, then re-run this cell. "
        "Or run: !uv pip install -q \"haicai @ git+https://github.com/ob1-s/haicai.git#subdirectory=packages/haicai-engine\" --no-deps -e /content/haicai/environments/haicai_v1"
    ) from e

# Build tasksets (train / eval split by offset — no overlap)
train_ts = HaicaiTaskset(HaicaiConfig(num_tasks=num_tasks, offset=task_offset))
eval_ts  = HaicaiTaskset(HaicaiConfig(num_tasks=eval_tasks, offset=eval_offset))
train_tasks = list(train_ts)
eval_tasks_list = list(eval_ts)

# HF dataset for TRL: pre-rendered conversational prompts (non-thinking: reward expects only the three haiku lines)
def _record(task):
    # Pre-render to string to avoid TRL+transformers 5.x Jinja batch bug on T4 (TypeError list+str)
    prompt_str = tokenizer.apply_chat_template(
        [{"role":"system","content": "Um haiku tem três versos de 5, 7 e 5 sílabas poéticas — conta-se até a última sílaba tônica de cada verso e vogais entre palavras vizinhas podem fundir."},
         {"role":"user","content": f"""Escreva um haiku original em português brasileiro sobre: {task.data.tema}.
Exemplo:

Uma folha morta.
Um galho, no céu grisalho.
Fecho a minha porta.

Responda APENAS com as três linhas do haiku."""}],
        tokenize=False, add_generation_prompt=True, enable_thinking=enable_thinking)
    return {"prompt": prompt_str, "idx": task.data.idx, "tema": task.data.tema}

train_dataset = Dataset.from_list([_record(t) for t in train_tasks]).shuffle(seed=seed)
eval_dataset  = Dataset.from_list([_record(t) for t in eval_tasks_list])
TASK_BY_IDX = {t.data.idx: t for t in train_tasks + eval_tasks_list}

print(f"train {len(train_dataset)}  eval {len(eval_dataset)}  tema ex: {train_dataset[0]['tema']}")
print(train_dataset[0]["prompt"])

# Sanity: show intervals
for demo in ["Vento frio na rua\nFolhas secas no chão\nNoite cai sem lua", "bad\nlines\nhere"]:
    vs = [v.strip() for v in demo.splitlines() if v.strip()]
    if len(vs)==3:
        for v in vs: print(v, escandir(v))
    print("coverage", coverage(demo), "\n")


In [ ]:
#@title 5 — Verifiers v1 → TRL reward bridge (uses HaicaiTask.score, no duplication)
import asyncio, nest_asyncio
from collections import defaultdict
from verifiers.v1.trace import Trace, TraceTask, AgentInfo
from verifiers.v1.graph import MessageNode
from verifiers.v1.types import SystemMessage, UserMessage, AssistantMessage
from verifiers.v1.configs.agent import AgentConfig

nest_asyncio.apply()
metrics_buf = defaultdict(list)

def _extract_text(completion):
    if isinstance(completion, list):
        for m in reversed(completion):
            if isinstance(m, dict) and m.get("role") == "assistant":
                return m.get("content", "")
        return completion[-1].get("content", "") if completion and isinstance(completion[-1], dict) else str(completion)
    return str(completion)

def build_reward_funcs(task_by_idx):
    def haicai_reward(prompts, completions, **kwargs):
        idxs = kwargs.get("idx", [None]*len(prompts))
        pairs = []
        for idx, comp in zip(idxs, completions):
            task = task_by_idx.get(idx)
            if task is None:
                continue
            text = _extract_text(comp)
            trace = Trace(task=TraceTask(type="HaicaiTask", data=task.data),
                          agent=AgentInfo(config=AgentConfig()))
            trace.nodes.append(MessageNode(parent=None, message=SystemMessage(content=task.data.system_prompt), sampled=False))
            trace.nodes.append(MessageNode(parent=0, message=UserMessage(content=task.data.prompt), sampled=False))
            trace.nodes.append(MessageNode(parent=1, message=AssistantMessage(content=text), sampled=True))
            trace.ok = trace.is_completed = True
            pairs.append((task, trace))
        if not pairs:
            return [0.0]*len(prompts)
        async def _score():
            await asyncio.gather(*(task.score(trace, runtime=None) for task, trace in pairs))
        asyncio.get_event_loop().run_until_complete(_score())
        rewards = []
        for _, trace in pairs:
            rewards.append(float(trace.reward))
            for k, r in trace.rewards.items():
                metrics_buf[f"rewards/{k}"].append(float(r.value) if r else 0.0)
            for k, v in trace.metrics.items():
                metrics_buf[f"env_metrics/{k}"].append(float(v) if v is not None else 0.0)
        return rewards
    haicai_reward.__name__ = "haicai_reward"
    return [haicai_reward]

trl_reward_funcs = build_reward_funcs(TASK_BY_IDX)

# Smoke — proves separation before training starts
print(trl_reward_funcs[0](
    prompts=[[{"role":"user","content":"x"}]]*3,
    completions=["Vento frio na rua\nFolhas secas no chão\nNoite cai sem lua", "bad", "Café quente na manhã\nO gato dorme na janela\nChuva fina lá fora"],
    idx=[train_tasks[0].data.idx]*3))
print("metrics_buf", dict(metrics_buf))
metrics_buf.clear()


In [ ]:
#@title 6 — GRPO config (T4-tuned)
from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    temperature=temperature, top_p=0.95, top_k=64,
    learning_rate=learning_rate, weight_decay=0.001, warmup_ratio=0.1,
    lr_scheduler_type="linear", optim="adamw_8bit",
    logging_steps=1, per_device_train_batch_size=1, gradient_accumulation_steps=2,
    num_generations=rollouts_per_sample, max_completion_length=max_completion_length,
    max_prompt_length=512, max_steps=max_steps, save_steps=save_steps,
    report_to="wandb" if use_wandb else "none", output_dir="outputs",
    beta=0.0, epsilon=0.2, epsilon_high=0.28, delta=1.5, loss_type="bnpo",
    mask_truncated_completions=True, log_completions=True,
)
print(training_args)


In [ ]:
#@title 7.-1 — Generation smoke (pre-RL checkpoint, LoRA off)
sample = train_dataset[0]
prompt = sample["prompt"]

if isinstance(prompt, str):
    inputs = tokenizer(text=prompt, return_tensors="pt")
else:
    inputs = tokenizer.apply_chat_template(prompt, add_generation_prompt=True, tokenize=True, enable_thinking=enable_thinking, return_tensors="pt")

if torch.cuda.is_available():
    inputs = {k: v.to("cuda") for k, v in inputs.items()}

from transformers import TextStreamer
streamer = TextStreamer(tokenizer, skip_prompt=True)

print("Prompt:", prompt)

# loaded checkpoint with LoRA off
with model.disable_adapter():
    _ = model.generate(**inputs, max_new_tokens=max_completion_length, do_sample=True, streamer=streamer)

In [ ]:
#@title 7 — Train (GRPO) — the moonshot
import types
# Gemma-4 + PEFT ref-adapter fix (#4934) — no-op on Qwen
_orig_add, _orig_get = model.add_adapter, model.get_parameter
def _patched_add(self, name, *a, **kw):
    return None if name=="ref" else _orig_add(name, *a, **kw)
class _DD:
    def copy_(self, o): pass
class _DP: data=_DD()
def _patched_get(self, t):
    return _DP() if ".ref." in t else _orig_get(t)
model.add_adapter = types.MethodType(_patched_add, model)
model.get_parameter = types.MethodType(_patched_get, model)

from transformers import TrainerCallback
class _MetricsCB(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kw):
        if not metrics_buf or logs is None: return
        for k,v in list(metrics_buf.items()): logs[k]=sum(v)/len(v)
        metrics_buf.clear()

trainer = GRPOTrainer(model=model, processing_class=tokenizer,
                      reward_funcs=trl_reward_funcs, args=training_args,
                      train_dataset=train_dataset, callbacks=[_MetricsCB()])
model.add_adapter=_orig_add; model.get_parameter=_orig_get
trainer.train()


In [ ]:
#@title 8 — Generation smoke (pre-RL checkpoint, LoRA off)
sample = train_dataset[0]
prompt = sample["prompt"]
# prompt is pre-rendered str (cell 4) — processor is Qwen3VL, so pass as text=, not positional
if isinstance(prompt, str):
    inputs = tokenizer(text=prompt, return_tensors="pt")
else:
    inputs = tokenizer.apply_chat_template(prompt, add_generation_prompt=True, tokenize=True, return_tensors="pt")

if torch.cuda.is_available():
    inputs = {k: v.to("cuda") for k, v in inputs.items()}

from transformers import TextStreamer
streamer = TextStreamer(tokenizer, skip_prompt=True)

print("Prompt:", prompt)

# loaded checkpoint with LoRA off
with model.disable_adapter():
    _ = model.generate(**inputs, max_new_tokens=max_completion_length, do_sample=True, temperature=temperature, top_p=0.95, top_k=64, streamer=streamer)

In [ ]:
#@title 8.1 — Generation smoke (with LoRA)
sample = train_dataset[0]
prompt = sample["prompt"]
if isinstance(prompt, str):
    inputs = tokenizer(text=prompt, return_tensors="pt")
else:
    inputs = tokenizer.apply_chat_template(prompt, add_generation_prompt=True, tokenize=True, return_tensors="pt")

if torch.cuda.is_available():
    inputs = {k: v.to("cuda") for k, v in inputs.items()}

print("Prompt:", prompt)

streamer = TextStreamer(tokenizer, skip_prompt=True)
# LoRA on (default)
_ = model.generate(**inputs, max_new_tokens=max_completion_length, do_sample=True, temperature=temperature, top_p=0.95, top_k=64, streamer=streamer)

In [ ]:
#@title 9 — Save LoRA
output_dir = "haicai_v1_lora" #@param {type:"string"}
push_to_hub = False #@param {type:"boolean"}
hf_repo = "oliveirabruno01/haicai-v1-lora" #@param {type:"string"}
model.save_pretrained(output_dir); tokenizer.save_pretrained(output_dir)
print(f"Saved {output_dir}", __import__("os").listdir(output_dir)[:6])
if push_to_hub:
    model.push_to_hub(hf_repo, token=os.environ.get("HF_TOKEN"))
    tokenizer.push_to_hub(hf_repo, token=os.environ.get("HF_TOKEN"))


## Why this is clean
- No vendoring: `haicai` + `haicai-v1` are the source; notebook clones `https://github.com/ob1-s/haicai` and `pip install -e` — edit once, train everywhere.
- Reward is `HaicaiTask.score` via `Trace` — same code `eval`/`GEPA`/`RL` all share. No duplicated `forma` math.
- Smoke → scale: `num_tasks=64, G=4, steps=10` finishes in ~10 min; set `400/8/80` for a real run; eval split stays disjoint via `eval_offset`.
